# EDA Unprocessed Dataset
Notebook ini membaca `dataset_absa_santika_raw.csv` hasil merge. Tidak ada preprocessing dan tidak ada WordCloud di tahap ini.


In [ ]:
from pathlib import Path
import pandas as pd
PROJECT_LOCAL_ROOT = Path(r'C:\Users\cencen04_\Downloads\ABSA Hotel Santika')
KAGGLE_INPUT = Path('/kaggle/input')
KAGGLE_WORKING = Path('/kaggle/working')
IS_KAGGLE = KAGGLE_INPUT.exists() and KAGGLE_WORKING.exists()
OUTPUT_DIR = KAGGLE_WORKING if IS_KAGGLE else PROJECT_LOCAL_ROOT / 'Data Preprocessing'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

def resolve_csv(filename, keywords=()):
    direct_candidates = [
        OUTPUT_DIR / filename,
        PROJECT_LOCAL_ROOT / 'Data Preprocessing' / filename,
        PROJECT_LOCAL_ROOT / 'Data Preprocessing' / 'Merge dan Wordcloud' / filename,
    ]
    for path in direct_candidates:
        if path.exists():
            print('Loaded:', path)
            return path

    candidates = []
    search_roots = [KAGGLE_INPUT, KAGGLE_WORKING, PROJECT_LOCAL_ROOT / 'Data Preprocessing', Path.cwd()]
    for root in search_roots:
        root = Path(root)
        if root.exists():
            candidates.extend(root.rglob(filename))

    def score(p):
        low = str(p).lower()
        value = sum(str(k).lower() in low for k in keywords)
        if filename.lower() in low:
            value += 2
        return value

    candidates = sorted(set(candidates), key=lambda p: (-score(p), str(p).lower()))
    if not candidates:
        raise FileNotFoundError(f'Tidak menemukan {filename}. Jalankan data_preprocessing.ipynb dulu atau upload outputnya ke Kaggle.')
    print('Loaded:', candidates[0])
    return candidates[0]


In [ ]:
RAW_PATH = resolve_csv('dataset_absa_santika_raw.csv', keywords=('raw', 'unprocessed'))
df = pd.read_csv(RAW_PATH, encoding='utf-8-sig', dtype=str)
EXPECTED_RAW_ROWS = 17868
print(f'Total review raw: {len(df):,}')
print(f'Kolom: {df.columns.tolist()}')
if len(df) != EXPECTED_RAW_ROWS:
    print(f'[WARN] Expected {EXPECTED_RAW_ROWS:,}, actual {len(df):,}')
display(df.head())


In [ ]:
print('=== Overview Raw Dataset ===')
print(f'Platform: {df["platform"].nunique()}')
print(f'Hotel: {df["hotel_name"].nunique()}')
dates = pd.to_datetime(df['date'], errors='coerce')
print(f'Rentang tanggal: {dates.min().date()} s/d {dates.max().date()}')
print(f'Tanggal invalid: {dates.isna().sum():,}')
print('\nMissing values:')
display(df.isna().sum().to_frame('missing_count'))


In [ ]:
platform_summary = df.groupby('platform').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
hotel_summary = df.groupby('hotel_name').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
platform_hotel = pd.crosstab(df['hotel_name'], df['platform'], margins=True)
print('--- Distribusi Platform ---'); display(platform_summary)
print('--- Distribusi Hotel ---'); display(hotel_summary)
print('--- Platform x Hotel ---'); display(platform_hotel)
if 'original_language' in df.columns:
    language_summary = df.groupby('original_language').size().to_frame('jumlah_review').reset_index().sort_values('jumlah_review', ascending=False)
    print('--- Bahasa Asli ---'); display(language_summary)


In [ ]:
text_len = df['text_review'].fillna('').astype(str).str.len()
quality_summary = pd.DataFrame({
    'metric': ['min_len', 'mean_len', 'median_len', 'max_len', 'duplicate_text', 'short_review_lt_20', 'empty_text'],
    'value': [text_len.min(), round(text_len.mean(), 2), text_len.median(), text_len.max(), df['text_review'].duplicated().sum(), (text_len < 20).sum(), df['text_review'].fillna('').astype(str).str.strip().eq('').sum()]
})
display(quality_summary)


In [ ]:
import matplotlib.pyplot as plt
years = pd.to_datetime(df['date'], errors='coerce').dt.year.value_counts().sort_index()
ax = years.plot(kind='bar', figsize=(12, 5), color='#4472C4')
ax.set_title('Distribusi Review per Tahun - Unprocessed')
ax.set_xlabel('Tahun'); ax.set_ylabel('Jumlah Review')
plt.tight_layout(); plt.show()
ax = platform_summary.set_index('platform')['jumlah_review'].plot(kind='bar', figsize=(8, 4), color='#70AD47')
ax.set_title('Distribusi Review per Platform - Unprocessed')
ax.set_xlabel('Platform'); ax.set_ylabel('Jumlah Review')
plt.tight_layout(); plt.show()


In [ ]:
summary_path = OUTPUT_DIR / 'eda_unprocessed_summary.xlsx'
with pd.ExcelWriter(summary_path) as writer:
    platform_summary.to_excel(writer, index=False, sheet_name='platform')
    hotel_summary.to_excel(writer, index=False, sheet_name='hotel')
    platform_hotel.to_excel(writer, sheet_name='platform_x_hotel')
    quality_summary.to_excel(writer, index=False, sheet_name='quality')
    if 'language_summary' in globals(): language_summary.to_excel(writer, index=False, sheet_name='language')
print('Saved:', summary_path)
